In [1]:
pip install pandas scikit-learn


In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load the dataset
data = pd.read_csv('/content/drive/My Drive/Colab Notebooks/ZOD-CSV-Datasets/PL_ZOD_Data.csv')

# Sort data by timestamp
data = data.sort_values(by='Time').reset_index(drop=True)

# Define speed ranges
def categorize_speed(speed):
    if speed < 10:
        return '0-10'
    elif speed < 20:
        return '10-20'
    elif speed < 30:
        return '20-30'
    elif speed < 40:
        return '30-40'
    elif speed < 50:
        return '40-50'
    elif speed < 60:
        return '50-60'
    elif speed < 70:
        return '60-70'
    elif speed < 80:
        return '70-80'
    elif speed < 90:
        return '80-90'
    elif speed < 100:
        return '90-100'
    elif speed < 110:
        return '100-110'
    elif speed < 120:
        return '110-120'
    elif speed < 130:
        return '120-130'
    else:
        return '130+'

# Apply speed categorization
data['Speed_Category'] = data['Speed (kph)'].apply(categorize_speed)

# Create next frame features
next_frame_cols = ['Number of Vehicles', 'Number of Lane Instances', 'Number of Pedestrians', 'Number of Traffic Lights', 'Number of Traffic Signs', 'Speed (kph)', 'Speed_Category']
for col in next_frame_cols:
    data[f'{col}_next'] = data[col].shift(-1)

# Drop the last row with NaN values for next frame features
data = data.dropna()

# Extract relevant features and target for training
features = ['Number of Vehicles', 'Number of Lane Instances', 'Number of Pedestrians', 'Number of Traffic Lights', 'Number of Traffic Signs', 'Weather Condition', 'Road Condition', 'Time of Day', 'Road Type']
next_frame_features = [f'{col}_next' for col in next_frame_cols[:-2]]  # Exclude 'Speed_Category_next' & 'Speed (kph)_next'
all_features = features + next_frame_features

X = data[all_features]
y = data['Speed_Category_next']

# Encode categorical variables
le_weather = LabelEncoder()
le_road_cond = LabelEncoder()
le_time_of_day = LabelEncoder()
le_road_type = LabelEncoder()

X['Weather Condition'] = le_weather.fit_transform(X['Weather Condition'])
X['Road Condition'] = le_road_cond.fit_transform(X['Road Condition'])
X['Time of Day'] = le_time_of_day.fit_transform(X['Time of Day'])
X['Road Type'] = le_road_type.fit_transform(X['Road Type'])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfc.fit(X_train, y_train)

# Make predictions
y_pred = rfc.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Display feature importances
importances = pd.Series(rfc.feature_importances_, index=all_features).sort_values(ascending=False)
print(importances)


<ipython-input-45-32ea25cf2c8d>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Weather Condition'] = le_weather.fit_transform(X['Weather Condition'])
<ipython-input-45-32ea25cf2c8d>:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Road Condition'] = le_road_cond.fit_transform(X['Road Condition'])
<ipython-input-45-32ea25cf2c8d>:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

              precision    recall  f1-score   support

        0-10       0.36      0.26      0.30       603
       10-20       0.25      0.05      0.08       449
     100-110       0.50      0.39      0.44        18
     110-120       0.27      0.33      0.30        12
     120-130       0.35      0.41      0.38        22
       20-30       0.27      0.27      0.27       849
       30-40       0.27      0.32      0.30      1151
       40-50       0.29      0.44      0.35      1206
       50-60       0.27      0.25      0.26       832
       60-70       0.25      0.19      0.22       426
       70-80       0.16      0.09      0.12       156
       80-90       0.29      0.03      0.06        61
      90-100       0.25      0.03      0.06        32

    accuracy                           0.28      5817
   macro avg       0.29      0.24      0.24      5817
weighted avg       0.28      0.28      0.27      5817

[[155  12   0   0   0 115 162 135  16   6   2   0   0]
 [ 44  22   0   0   0 12

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load the dataset
data = pd.read_csv('/content/drive/My Drive/Colab Notebooks/ZOD-CSV-Datasets/PL_ZOD_Data.csv')

# Sort data by timestamp
data = data.sort_values(by='Time').reset_index(drop=True)

# Create lag features (previous frame)
for col in ['Number of Vehicles', 'Number of Lane Instances', 'Number of Pedestrians', 'Number of Traffic Lights', 'Number of Traffic Signs', 'Speed (kph)']:
    data[f'{col}_prev'] = data[col].shift(1)

# Create difference features
for col in ['Number of Vehicles', 'Number of Lane Instances', 'Number of Pedestrians', 'Number of Traffic Lights', 'Number of Traffic Signs', 'Speed (kph)']:
    data[f'{col}_diff'] = data[col] - data[f'{col}_prev']

# Drop the first row with NaN values from lag feature creation
data = data.dropna().reset_index(drop=True)

print(data.head())

# Encode categorical variables
categorical_cols = ['Weather Condition', 'Road Condition', 'Time of Day', 'Road Type']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# Define speed ranges
def speed_range(speed):
    if speed < 10:
        return '0-10'
    elif speed < 20:
        return '10-20'
    elif speed < 30:
        return '20-30'
    elif speed < 40:
        return '30-40'
    elif speed < 50:
        return '40-50'
    elif speed < 60:
        return '50-60'
    elif speed < 70:
        return '60-70'
    elif speed < 80:
        return '70-80'
    elif speed < 90:
        return '80-90'
    elif speed < 100:
        return '90-100'
    elif speed < 110:
        return '100-110'
    elif speed < 120:
        return '110-120'
    elif speed < 130:
        return '120-130'
    elif speed < 140:
        return '130-140'
    elif speed < 150:
        return '140-150'
    else:
        return '150+'


# Apply speed range function to create target labels
data['Speed Range'] = data['Speed (kph)'].apply(speed_range)

# Features and target variable
features = ['Number of Vehicles', 'Number of Lane Instances', 'Number of Pedestrians', 'Number of Traffic Lights',
            'Number of Traffic Signs', 'Speed (kph)_prev', 'Number of Vehicles_diff', 'Number of Lane Instances_diff',
            'Number of Pedestrians_diff', 'Number of Traffic Lights_diff', 'Number of Traffic Signs_diff'] + categorical_cols
target = 'Speed Range'

# Split the data into train and test sets
X = data[features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Feature importance
feature_importances = pd.DataFrame(clf.feature_importances_, index=features, columns=['importance']).sort_values('importance', ascending=False)
print(feature_importances)


   Frame ID Country Code  Longitude   Latitude  \
0     64118           PL  19.054200  50.266932   
1     94286           PL  19.111246  50.311502   
2     26317           PL  19.110567  50.313415   
3     90989           PL  19.107478  50.328272   
4     72276           PL  19.043326  50.262459   

                               Time  Weather Condition Road Condition  \
0  2021-03-26 08:37:26.774423+00:00  partly-cloudy-day         normal   
1  2021-03-26 08:48:26.883220+00:00  partly-cloudy-day         normal   
2  2021-03-26 08:48:48.216753+00:00  partly-cloudy-day         normal   
3  2021-03-26 09:16:17.106787+00:00  partly-cloudy-day         normal   
4  2021-03-26 09:25:25.114849+00:00  partly-cloudy-day         normal   

  Time of Day       Road Type  Number of Vehicles  ...  \
0         day  arterial-urban                  15  ...   
1         day  arterial-rural                  18  ...   
2         day  arterial-urban                  10  ...   
3         day  arterial-urba

In [11]:
!pip install tensorflow

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix

# Load the dataset
data = pd.read_csv('/content/drive/My Drive/Colab Notebooks/ZOD-CSV-Datasets/SE_ZOD_Data.csv')

# Sort data by timestamp
data = data.sort_values(by='Time').reset_index(drop=True)

# Create lag features (previous frame)
for col in ['Number of Vehicles', 'Number of Lane Instances', 'Number of Pedestrians', 'Number of Traffic Lights', 'Number of Traffic Signs', 'Speed (kph)']:
    data[f'{col}_prev'] = data[col].shift(1)

# Create difference features
for col in ['Number of Vehicles', 'Number of Lane Instances', 'Number of Pedestrians', 'Number of Traffic Lights', 'Number of Traffic Signs', 'Speed (kph)']:
    data[f'{col}_diff'] = data[col] - data[f'{col}_prev']

# Drop the first row with NaN values from lag feature creation
data = data.dropna().reset_index(drop=True)

# Encode categorical variables
categorical_cols = ['Weather Condition', 'Road Condition', 'Time of Day', 'Road Type']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# Define speed ranges
def speed_range(speed):
    if speed < 10:
        return 0
    elif speed < 20:
        return 1
    elif speed < 30:
        return 2
    elif speed < 40:
        return 3
    elif speed < 50:
        return 4
    elif speed < 60:
        return 5
    elif speed < 70:
        return 6
    elif speed < 80:
        return 7
    elif speed < 90:
        return 8
    elif speed < 100:
        return 9
    elif speed < 110:
        return 10
    elif speed < 120:
        return 11
    elif speed < 130:
        return 12
    elif speed < 140:
        return 13
    elif speed < 150:
        return 14
    else:
        return 15

# Apply speed range function to create target labels
data['Speed Range'] = data['Speed (kph)'].apply(speed_range)

# Features and target variable
features = ['Number of Vehicles', 'Number of Lane Instances', 'Number of Pedestrians', 'Number of Traffic Lights',
            'Number of Traffic Signs', 'Speed (kph)_prev', 'Number of Vehicles_diff', 'Number of Lane Instances_diff',
            'Number of Pedestrians_diff', 'Number of Traffic Lights_diff', 'Number of Traffic Signs_diff'] + categorical_cols
target = 'Speed Range'

# Split the data into train and test sets
X = data[features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert target variable to categorical
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

# Build the neural network
model = Sequential()
model.add(Dense(128, input_dim=X_train.shape[1], activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(y_train.shape[1], activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

print(classification_report(y_test_classes, y_pred_classes))
print(confusion_matrix(y_test_classes, y_pred_classes))


Epoch 1/50
338/338 [==============================] - 5s 7ms/step - loss: 1.9714 - accuracy: 0.2585 - val_loss: 1.6728 - val_accuracy: 0.3510
Epoch 2/50
338/338 [==============================] - 1s 3ms/step - loss: 1.7462 - accuracy: 0.3022 - val_loss: 1.6167 - val_accuracy: 0.3551
Epoch 3/50
338/338 [==============================] - 1s 3ms/step - loss: 1.6901 - accuracy: 0.3285 - val_loss: 1.5933 - val_accuracy: 0.3795
Epoch 4/50
338/338 [==============================] - 1s 3ms/step - loss: 1.6579 - accuracy: 0.3427 - val_loss: 1.5631 - val_accuracy: 0.3947
Epoch 5/50
338/338 [==============================] - 1s 3ms/step - loss: 1.6353 - accuracy: 0.3473 - val_loss: 1.5496 - val_accuracy: 0.4010
Epoch 6/50
338/338 [==============================] - 1s 3ms/step - loss: 1.6191 - accuracy: 0.3614 - val_loss: 1.5316 - val_accuracy: 0.4043
Epoch 7/50
338/338 [==============================] - 1s 3ms/step - loss: 1.6009 - accuracy: 0.3681 - val_loss: 1.5291 - val_accuracy: 0.4143
Epoch 

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.

In [ ]:
import pandas as pd

# Load the CSV file
file_path = '/content/drive/My Drive/Colab Notebooks/PL_ZOD_Data.csv'
data = pd.read_csv(file_path)

# Display the first few rows of the dataframe
data.head()


,Frame ID,Country Code,Longitude,Latitude,Time,Weather Condition,Road Condition,Time of Day,Road Type,Number of Vehicles,Number of Lane Instances,Number of Pedestrians,Number of Traffic Lights,Number of Traffic Signs,Speed (kph),next_frame_id,distance_to_next
0,48035,PL,19.010398,50.267241,2021-03-26 08:33:48.004884+00:00,partly-cloudy-day,normal,day,city,35,5,5,29,52,35.972596,37380,0.002771
1,64118,PL,19.054200,50.266932,2021-03-26 08:37:26.774423+00:00,partly-cloudy-day,normal,day,arterial-urban,15,7,0,0,19,70.555278,99828,0.005885
2,94286,PL,19.111246,50.311502,2021-03-26 08:48:26.883220+00:00,partly-cloudy-day,normal,day,arterial-rural,18,7,0,8,10,35.674392,26317,0.002031
3,26317,PL,19.110567,50.313415,2021-03-26 08:48:48.216753+00:00,partly-cloudy-day,normal,day,arterial-urban,10,7,0,17,17,4.990055,94286,0.002031
4,90989,PL,19.107478,50.328272,2021-03-26 09:16:17.106787+00:00,partly-cloudy-day,normal,day,arterial-urban,4,3,0,0,1,68.665545,89257,0.006284


In [ ]:
import pandas as pd

# Step 1: Sort by Time
data.sort_values(by='Time', inplace=True)

# Step 2: Calculate differences between consecutive rows
data['speed_change'] = data['Speed (kph)'].diff()
data['number_of_vehicles_change'] = data['Number of Vehicles'].diff()
data['number_of_lane_instances_change'] = data['Number of Lane Instances'].diff()
data['number_of_pedestrians_change'] = data['Number of Pedestrians'].diff()
data['number_of_traffic_lights_change'] = data['Number of Traffic Lights'].diff()
data['number_of_traffic_signs_change'] = data['Number of Traffic Signs'].diff()

# Step 3: Encode categorical variables
data['weather_change'] = (data['Weather Condition'] != data['Weather Condition'].shift()).astype(int)
data['road_condition_change'] = (data['Road Condition'] != data['Road Condition'].shift()).astype(int)
data['time_of_day_change'] = (data['Time of Day'] != data['Time of Day'].shift()).astype(int)
data['road_type_change'] = (data['Road Type'] != data['Road Type'].shift()).astype(int)

# Step 4: One-hot encode categorical features
data = pd.get_dummies(data, columns=['weather_change', 'road_condition_change', 'time_of_day_change', 'road_type_change'])

# Step 5: Select relevant features in the specified order
features = [
    'number_of_vehicles_change',
    'number_of_lane_instances_change', 'number_of_pedestrians_change',
    'number_of_traffic_lights_change', 'number_of_traffic_signs_change',
    'weather_change_0', 'weather_change_1',
    'road_condition_change_0', 'road_condition_change_1',
    'time_of_day_change_0', 'time_of_day_change_1',
    'road_type_change_0', 'road_type_change_1'
]

# Drop NaN values resulting from diff() operation in the first row
data.dropna(inplace=True)

# Display the first few rows of the processed data and selected features
selected_data = data[features]
selected_data.head()


,number_of_vehicles_change,number_of_lane_instances_change,number_of_pedestrians_change,number_of_traffic_lights_change,number_of_traffic_signs_change,weather_change_0,weather_change_0,weather_change_0,weather_change_1,weather_change_1,...,time_of_day_change_0,time_of_day_change_1,time_of_day_change_1,time_of_day_change_1,road_type_change_0,road_type_change_0,road_type_change_0,road_type_change_1,road_type_change_1,road_type_change_1
3,-8.0,0.0,0.0,9.0,7.0,True,True,True,False,False,...,True,False,False,False,False,False,False,True,True,True
4,-6.0,-4.0,0.0,-17.0,-16.0,True,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
5,11.0,7.0,1.0,0.0,32.0,True,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
6,-5.0,-6.0,-1.0,9.0,0.0,True,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
7,16.0,6.0,3.0,19.0,19.0,True,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False


In [ ]:
# Calculate the difference in speed between consecutive frames
merged_data['speed_change'] = merged_data['Speed (kph)_end'] - merged_data['Speed (kph)_start']

# Calculate the difference for other relevant features
merged_data['longitude_change'] = merged_data['Longitude_end'] - merged_data['Longitude_start']
merged_data['latitude_change'] = merged_data['Latitude_end'] - merged_data['Latitude_start']
merged_data['number_of_vehicles_change'] = merged_data['Number of Vehicles_end'] - merged_data['Number of Vehicles_start']
merged_data['number_of_lane_instances_change'] = merged_data['Number of Lane Instances_end'] - merged_data['Number of Lane Instances_start']
merged_data['number_of_pedestrians_change'] = merged_data['Number of Pedestrians_end'] - merged_data['Number of Pedestrians_start']
merged_data['number_of_traffic_lights_change'] = merged_data['Number of Traffic Lights_end'] - merged_data['Number of Traffic Lights_start']
merged_data['number_of_traffic_signs_change'] = merged_data['Number of Traffic Signs_end'] - merged_data['Number of Traffic Signs_start']

# Encode categorical variables
merged_data['weather_change'] = merged_data['Weather Condition_end'] != merged_data['Weather Condition_start']
merged_data['road_condition_change'] = merged_data['Road Condition_end'] != merged_data['Road Condition_start']
merged_data['time_of_day_change'] = merged_data['Time of Day_end'] != merged_data['Time of Day_start']
merged_data['road_type_change'] = merged_data['Road Type_end'] != merged_data['Road Type_start']

# One-hot encode the categorical features
merged_data = pd.get_dummies(merged_data, columns=['weather_change', 'road_condition_change', 'time_of_day_change', 'road_type_change'])

# Select the relevant features for modeling
features = [
    'longitude_change', 'latitude_change', 'number_of_vehicles_change',
    'number_of_lane_instances_change', 'number_of_pedestrians_change',
    'number_of_traffic_lights_change', 'number_of_traffic_signs_change'
] + [col for col in merged_data.columns if 'weather_change' in col or 'road_condition_change' in col or 'time_of_day_change' in col or 'road_type_change' in col]


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Split the data into training and testing sets
X = data[features]
y = data['speed_change']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')


Mean Squared Error: 317.40177370768544


In [ ]:
importances = model.feature_importances_
feature_importance = pd.DataFrame({'Feature': features, 'Importance': importances})
feature_importance.sort_values(by='Importance', ascending=False, inplace=True)

# Display feature importance
feature_importance


ValueError: All arrays must be of the same length

In [ ]:
import os

path = '/content/drive/My Drive/Colab Notebooks/ZODFullDataset/Frames/single_frames/000780'

# Check if the path exists
if os.path.exists(path):
    # Use os.walk to navigate through the directory
    folder_count = sum(os.path.isdir(os.path.join(path, entry)) for entry in os.listdir(path))
    print(f"Number of folders in '{path}': {folder_count}")
else:
    print("Specified path does not exist")


Number of folders in '/content/drive/My Drive/Colab Notebooks/ZODFullDataset/Frames/single_frames/000780': 1


In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

base_path = '/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames'

# Function to load metadata for all frames
def load_metadata(base_path):
    metadata = []
    for frame_id in os.listdir(base_path):
        frame_path = os.path.join(base_path, frame_id)
        if os.path.isdir(frame_path):
            metadata_path = os.path.join(frame_path, 'metadata.json')
            if os.path.exists(metadata_path):
                with open(metadata_path, 'r') as file:
                    data = json.load(file)
                    metadata.append(data)
    return pd.DataFrame(metadata)

# Filter rare instances and non-peak times
def filter_metadata(metadata):
    # Omit rows with rare weather and road conditions
    metadata = metadata[(metadata['scraped_weather'] != 'snow') & (metadata['road_condition'] != 'snow')]

    # Convert 'Time' to datetime
    metadata['time'] = pd.to_datetime(metadata['time'])

    # Define peak hours
    start_time = datetime.strptime("07:00:00", "%H:%M:%S").time()
    end_time = datetime.strptime("19:00:00", "%H:%M:%S").time()

    # Omit rows outside of peak times
    metadata = metadata[(metadata['time'].dt.time >= start_time) & (metadata['time'].dt.time <= end_time)]

    return metadata

# Load and filter metadata
metadata = load_metadata(base_path)
filtered_metadata = filter_metadata(metadata)

# Print the head of the filtered metadata
print(filtered_metadata.head())


  frame_id                             time country_code    scraped_weather  \
0   009158 2020-11-21 09:44:55.445654+00:00           DE             cloudy   
1   018591 2020-07-24 07:08:56.778861+00:00           IT               rain   
2   023996 2021-04-15 14:52:36.885711+00:00           PL               rain   
3   029229 2021-05-02 14:22:06.780681+00:00           PL               rain   
4   044953 2020-05-06 11:41:57.779576+00:00           SE  partly-cloudy-day   

  collection_car road_type road_condition time_of_day  num_lane_instances  \
0          india      city         normal         day                   1   
1          india      city         normal         day                   3   
2          india      city            wet         day                   6   
3           golf      city            wet         day                  10   
4          india      city         normal         day                   0   

   num_vehicles  num_vulnerable_vehicles  num_pedestrians  num

<ipython-input-3-f7cf916b7e96>:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['time'] = pd.to_datetime(metadata['time'])


In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

base_path = '/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames'

# Function to load metadata for all frames
def load_metadata(base_path):
    metadata = []
    for frame_id in os.listdir(base_path):
        frame_path = os.path.join(base_path, frame_id)
        if os.path.isdir(frame_path):
            metadata_path = os.path.join(frame_path, 'metadata.json')
            if os.path.exists(metadata_path):
                with open(metadata_path, 'r') as file:
                    data = json.load(file)
                    metadata.append(data)
    return pd.DataFrame(metadata)

# Filter rare instances and non-peak times
def filter_metadata(metadata, peak_hours):
    # Omit rows with rare weather and road conditions
    metadata = metadata[(metadata['scraped_weather'] != 'snow') & (metadata['road_condition'] != 'snow')]

    # Convert 'Time' to datetime
    metadata['time'] = pd.to_datetime(metadata['time'])

    # Filter based on peak hours
    filtered_metadata = metadata[metadata['time'].dt.time.apply(lambda t: any(start <= t <= end for start, end in peak_hours))]

    return filtered_metadata

# Load metadata
metadata = load_metadata(base_path)

# Define peak hours
peak_hours_1 = [(datetime.strptime("07:00:00", "%H:%M:%S").time(), datetime.strptime("19:00:00", "%H:%M:%S").time())]
peak_hours_2 = [
    (datetime.strptime("06:30:00", "%H:%M:%S").time(), datetime.strptime("09:30:00", "%H:%M:%S").time()),
    (datetime.strptime("16:30:00", "%H:%M:%S").time(), datetime.strptime("19:00:00", "%H:%M:%S").time())
]

# Filter metadata for both sets of peak hours
filtered_metadata_1 = filter_metadata(metadata, peak_hours_1)
filtered_metadata_2 = filter_metadata(metadata, peak_hours_2)

# Save to CSV files
filtered_metadata_1.to_csv('/content/drive/My Drive/Colab Notebooks/filtered_metadata_07_00_to_19_00.csv', index=False)
filtered_metadata_2.to_csv('/content/drive/My Drive/Colab Notebooks/filtered_metadata_06_30_to_09_30_and_16_30_to_19_00.csv', index=False)

# Print confirmation message
print("CSV files have been saved.")


CSV files have been saved.


<ipython-input-6-d3022add2390>:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['time'] = pd.to_datetime(metadata['time'])
<ipython-input-6-d3022add2390>:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['time'] = pd.to_datetime(metadata['time'])


In [ ]:
import cv2
import torch
import json
import os
import matplotlib.pyplot as plt

# Load a pre-trained YOLOv5 model (latest version)
model = torch.hub.load('ultralytics/yolov5', 'yolov5x')

# Function to detect objects in an image
def detect_objects(image_path):
    for file_name in os.listdir(image_path):
        if file_name.endswith('.jpg'):
            img_path = os.path.join(image_path, file_name)
            img = cv2.imread(img_path)
            results = model(img)
            return results.pandas().xyxy[0]
    return None

# Extract features from an image
def extract_annotations(annotation_path):
    with open(annotation_path, 'r') as file:
        annotations = json.load(file)
    return annotations

# Extract features from an image using both detections and annotations
def extract_features(image_path):
    # Detect objects using YOLO
    detections = detect_objects(image_path)
    if detections is None:
        print(f'No image file found in {image_path}')
        return None

    # Extract relevant annotation data
    vehicles = len(detections[detections['name'] == 'car'])
    pedestrians = len(detections[detections['name'] == 'person'])
    traffic_lights = len(detections[detections['name'] == 'traffic light'])

    # Combine detection results with annotations
    features = {
        'vehicles': vehicles,
        'pedestrians': pedestrians,
        'traffic_lights': traffic_lights
    }

    return features

# List of frame IDs
frame_ids = ['007674', '009158', '018591', '023996', '029229', '044953', '062592', '065838', '070221', '082291', '083430', '087912']

# Base path to the dataset
base_path = '/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames'

# Dictionary to store features for each frame
all_features = {}

for frame_id in frame_ids:
    image_path = os.path.join(base_path, frame_id, 'camera_front_blur')
    features = extract_features(image_path)
    if features is not None:
        all_features[frame_id] = features

print(all_features)


/usr/local/lib/python3.10/dist-packages/torch/hub.py:293: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip


requirements: Ultralytics requirements ['gitpython>=3.1.30', 'pillow>=10.3.0', 'requests>=2.32.0'] not found, attempting AutoUpdate...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 142.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 117.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pillow
    Found existing installation: Pillow 9.4.0
    Uninstalling Pillow-9.4.0:
      Successfully uninstalled Pillow-9.4.0

requirements: AutoUpdate success ✅ 11.8s, installed 3 packages: ['gitpython>=3.1.30', 'pillow>=10.3.0', 'requests>=2.32.0']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect



YOLOv5 🚀 2024-7-13 Python-3.10.12 torch-2.3.0+cu121 CPU

100%|██████████| 166M/166M [00:00<00:00, 251MB/s]

Fusing layers... 
YOLOv5x summary: 444 layers, 86705005 parameters, 0 gradients, 205.5 GFLOPs
Adding AutoShape... 


{'007674': {'vehicles': 14, 'pedestrians': 0, 'traffic_lights': 0}, '009158': {'vehicles': 10, 'pedestrians': 0, 'traffic_lights': 0}, '018591': {'vehicles': 8, 'pedestrians': 2, 'traffic_lights': 3}, '023996': {'vehicles': 5, 'pedestrians': 1, 'traffic_lights': 1}, '029229': {'vehicles': 8, 'pedestrians': 0, 'traffic_lights': 0}, '044953': {'vehicles': 2, 'pedestrians': 1, 'traffic_lights': 0}, '062592': {'vehicles': 0, 'pedestrians': 1, 'traffic_lights': 0}, '065838': {'vehicles': 15, 'pedestrians': 1, 'traffic_lights': 5}, '070221': {'vehicles': 6, 'pedestrians': 1, 'traffic_lights': 0}, '082291': {'vehicles': 4, 'pedestrians': 0, 'traffic_lights': 0}, '083430': {'vehicles': 2, 'pedestrians': 0, 'traffic_lights': 7}, '087912': {'vehicles': 8, 'pedestrians': 0, 'traffic_lights': 0}}


In [ ]:
import cv2
import torch
import json
import os

# Load a pre-trained YOLO model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')

def detect_objects(image_path):
    for file_name in os.listdir(image_path):
        if file_name.endswith('.jpg'):
            img = cv2.imread(os.path.join(image_path, file_name))
            results = model(img)
            return results.pandas().xyxy[0]
    return None

def extract_annotations(annotation_path):
    traffic_signs_path = os.path.join(annotation_path, 'object_detection.json')
    with open(traffic_signs_path, 'r') as file:
        annotations = json.load(file)

    # Filter annotations based on relevance to ego road and lane
    relevant_annotations = []
    for annotation in annotations:
        properties = annotation['properties']

        # Check conditions for relevance
        if (properties.get('is_for_ego_road') != 'False' and
            properties.get('is_for_ego_lane') != 'False' and
            properties.get('relative_position') in ['EgoLane', 'LeftAndEgoLane', 'RightAndEgoLane']):
            relevant_annotations.append(annotation)
        else:
            relevant_annotations.append(annotation)  # Include annotations where these properties are not defined

    return relevant_annotations


def count_objects(annotations, class_name):
    count = sum(1 for anno in annotations if anno['properties']['class'] == class_name)
    return count

def extract_features(image_path, annotation_path):
    # Load annotations
    annotations = extract_annotations(annotation_path)

    # Count relevant classes from annotations
    vehicle_count = count_objects(annotations, 'Vehicle')
    traffic_sign_count = count_objects(annotations, 'TrafficSign')
    traffic_signal_count = count_objects(annotations, 'TrafficSignal')
    pole_object_count = count_objects(annotations, 'PoleObject')
    pedestrian_count = count_objects(annotations, 'Pedestrian')
    vulnerable_vehicle_count = count_objects(annotations, 'VulnerableVehicle')
    traffic_guide_count = count_objects(annotations, 'TrafficGuide')

    # Combine counts
    features = {
        'vehicles': vehicle_count,
        'pedestrians': pedestrian_count,
        'traffic_signs': traffic_sign_count,
        'traffic_signals': traffic_signal_count,
        'pole_objects': pole_object_count,
        'vulnerable_vehicles': vulnerable_vehicle_count,
        'traffic_guides': traffic_guide_count
    }

    return features


# Example usage for multiple frames
frame_ids = ['007674', '009158', '018591', '023996', '029229', '044953', '062592', '065838', '070221', '082291', '083430', '087912']
base_path = '/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames'

for frame_id in frame_ids:
    image_path = os.path.join(base_path, frame_id, 'camera_front_blur')
    annotation_path = os.path.join(base_path, frame_id, 'annotations')
    features = extract_features(image_path, annotation_path)
    print(f'Frame {frame_id} features: {features}')


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-7-12 Python-3.10.12 torch-2.3.0+cu121 CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Frame 007674 features: {'vehicles': 23, 'pedestrians': 6, 'traffic_signs': 20, 'traffic_signals': 7, 'pole_objects': 11, 'vulnerable_vehicles': 11, 'traffic_guides': 2}
Frame 009158 features: {'vehicles': 39, 'pedestrians': 4, 'traffic_signs': 16, 'traffic_signals': 3, 'pole_objects': 9, 'vulnerable_vehicles': 9, 'traffic_guides': 0}
Frame 018591 features: {'vehicles': 29, 'pedestrians': 6, 'traffic_signs': 24, 'traffic_signals': 7, 'pole_objects': 21, 'vulnerable_vehicles': 4, 'traffic_guides': 0}
Frame 023996 features: {'vehicles': 20, 'pedestrians': 9, 'traffic_signs': 13, 'traffic_signals': 9, 'pole_objects': 27, 'vulnerable_vehicles': 2, 'traffic_guides': 66}
Frame 029229 features: {'vehicles': 24, 'pedestrians': 7, 'traffic_signs': 70, 'traffic_signals': 6, 'pole_objects': 0, 'vulnerable_vehicles': 3, 'traffic_guides': 4}
Frame 044953 features: {'vehicles': 8, 'pedestrians': 2, 'traffic_signs': 5, 'traffic_signals': 0, 'pole_objects': 14, 'vulnerable_vehicles': 1, 'traffic_guides

In [ ]:
import os
import json

def load_annotations(file_path):
    with open(file_path, 'r') as file:
        annotations = json.load(file)
    return annotations

def extract_annotation_details(annotations):
    details = []
    for annotation in annotations:
        properties = annotation.get('properties', {})

        detail = {
            "class": properties.get("class", "Unknown"),
            "unclear": properties.get("unclear", "Unknown"),
            "traffic_content_visible": properties.get("traffic_content_visible", "Unknown"),
            "is_for_construction": properties.get("is_for_construction", "Unknown"),
            "is_for_ego_road": properties.get("is_for_ego_road", "Unknown"),
            "is_for_ego_lane": properties.get("is_for_ego_lane", "Unknown"),
            "is_for_other_traffic_participant": properties.get("is_for_other_traffic_participant", "Unknown"),
            "is_on_dynamic_object": properties.get("is_on_dynamic_object", "Unknown"),
            "occlusion_ratio": properties.get("occlusion_ratio", "Unknown"),
            "size_3d_length": properties.get("size_3d_length", "Unknown"),
            "size_3d_width": properties.get("size_3d_width", "Unknown"),
            "size_3d_height": properties.get("size_3d_height", "Unknown"),
            "location_3d": properties.get("location_3d", {}).get("coordinates", "Unknown")
        }

        details.append(detail)

    return details

def print_annotation_details(details):
    for detail in details:
        print(f"Class: {detail['class']}")
        print(f"Unclear: {detail['unclear']}")
        print(f"Traffic Content Visible: {detail['traffic_content_visible']}")
        print(f"Is for Construction: {detail['is_for_construction']}")
        print(f"Is for Ego Road: {detail['is_for_ego_road']}")
        print(f"Is for Ego Lane: {detail['is_for_ego_lane']}")
        print(f"Is for Other Traffic Participant: {detail['is_for_other_traffic_participant']}")
        print(f"Is on Dynamic Object: {detail['is_on_dynamic_object']}")

        print('-' * 50)

def process_all_frames(frame_ids, base_path):
    for frame_id in frame_ids:
        annotation_file_path = os.path.join(base_path, frame_id, 'annotations', 'object_detection.json')
        if os.path.exists(annotation_file_path):
            print(f"Processing frame: {frame_id}")
            annotations = load_annotations(annotation_file_path)
            annotation_details = extract_annotation_details(annotations)
            print_annotation_details(annotation_details)
        else:
            print(f"Annotation file not found for frame: {frame_id}")

# Example usage
frame_ids = ['007674', '009158', '018591', '023996', '029229', '044953', '062592', '065838', '070221', '082291', '083430', '087912']
base_path = '/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames'

process_all_frames(frame_ids, base_path)



Streaming output truncated to the last 5000 lines.
Traffic Content Visible: True
Is for Construction: False
Is for Ego Road: True
Is for Ego Lane: True
Is for Other Traffic Participant: False
Is on Dynamic Object: False
--------------------------------------------------
Class: TrafficSign
Unclear: True
Traffic Content Visible: Unknown
Is for Construction: Unknown
Is for Ego Road: Unknown
Is for Ego Lane: Unknown
Is for Other Traffic Participant: Unknown
Is on Dynamic Object: Unknown
--------------------------------------------------
Class: TrafficSignal
Unclear: False
Traffic Content Visible: True
Is for Construction: False
Is for Ego Road: False
Is for Ego Lane: False
Is for Other Traffic Participant: False
Is on Dynamic Object: False
--------------------------------------------------
Class: TrafficSign
Unclear: True
Traffic Content Visible: Unknown
Is for Construction: Unknown
Is for Ego Road: Unknown
Is for Ego Lane: Unknown
Is for Other Traffic Participant: Unknown
Is on Dynamic Ob

In [ ]:
import json
import os
from collections import defaultdict

def extract_annotations(annotation_path):
    traffic_signs_path = os.path.join(annotation_path, 'object_detection.json')
    with open(traffic_signs_path, 'r') as file:
        annotations = json.load(file)
    return annotations

def get_unique_properties(annotations):
    unique_properties = defaultdict(set)
    for annotation in annotations:
        properties = annotation.get('properties', {})
        for key, value in properties.items():
            # Convert any dictionary values to a string
            if isinstance(value, dict):
                value = json.dumps(value)
            # Convert any list values to a tuple
            elif isinstance(value, list):
                value = tuple(value)
            unique_properties[key].add(value)
    return unique_properties

# Example usage for multiple frames
frame_ids = ['007674', '009158', '018591', '023996', '029229', '044953', '062592', '065838', '070221', '082291', '083430', '087912']
base_path = '/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames'

all_unique_properties = defaultdict(set)

for frame_id in frame_ids:
    annotation_path = os.path.join(base_path, frame_id, 'annotations')
    annotations = extract_annotations(annotation_path)
    unique_properties = get_unique_properties(annotations)
    for key, values in unique_properties.items():
        all_unique_properties[key].update(values)

# Print out all unique properties and their values
for key, values in all_unique_properties.items():
    print(f"Property: {key}")
    print(f"Values: {values}\n")


Property: annotation_uuid
Values: {'810174df-362c-4130-8811-07e9e6cb5c15', '755cc39e-21dd-4339-9aeb-43fd4d8a5896', '72f2b6d0-2a35-454a-92e9-ec3a884e50b4', '48e0638d-19a0-44e4-90ed-39b922c971b4', '21117e4c-b770-4841-a4de-aa0d5cac210c', 'e384de4c-ea0e-41d3-a63d-847a89d624ce', 'e78eef54-0d72-47bf-a9ef-6e28b83b45fe', '42b124f5-1fe2-4f4c-8063-dab7a169bf6d', '36ed78f2-5e13-4a27-bbf5-e974d9e596a4', '29913895-c96f-47f7-8b12-0fc186aec7bf', 'da21424b-07c5-4c9b-bf50-fcd36c8ac772', 'a05a3cce-c188-40b2-a87a-214f33557899', 'e433a6ca-2fc8-4d22-b732-1edaee79f549', '86a33fbb-5784-42ac-a740-14c5ba464217', 'd18a3f4a-4d3a-4888-ad3e-4c094f0f06bb', '7cbc6b23-35b5-48a5-a8e0-ca4a3de72e54', '7f2accd4-2c77-4927-bf41-284aa6a35294', '05c6d434-e2e5-4893-b8fe-6352d5cafc70', 'aa0d4dd2-4bb2-4634-b09f-b3c4978341e3', '1519bd9e-d96c-4036-9f08-886628ca718f', '6f7f6ad0-0857-4add-80ce-7ae6c880d3f6', 'f2094027-94e0-4f52-ad30-354d79db9d15', '0c1b6468-61dc-42cd-9895-a7974c073f3e', '4d759318-8260-4a8a-8fc7-1292896a8715', '87df

In [ ]:
!pip install opencv-python torch torchvision pandas


  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (99 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21

In [ ]:
import cv2
import torch

# Load a pre-trained YOLOv5 model (latest version)
model = torch.hub.load('ultralytics/yolov5', 'yolov5x')

def detect_objects(image_path):
    img = cv2.imread(image_path)
    results = model(img)
    return results.pandas().xyxy[0]

# Extract features from an image
def extract_features(image_path):
    detections = detect_objects(image_path)
    features = {
        'vehicles': len(detections[detections['name'] == 'car']),
        'pedestrians': len(detections[detections['name'] == 'person'])
    }
    return features

# Example usage
image_path = '/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames/087912/camera_front blur'
features = extract_features(image_path)
print(features)


/usr/local/lib/python3.10/dist-packages/torch/hub.py:293: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip


requirements: Ultralytics requirements ['gitpython>=3.1.30', 'pillow>=10.3.0', 'requests>=2.32.0'] not found, attempting AutoUpdate...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 67.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 113.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 130.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pillow
    Found existing installation: Pillow 9.4.0
    Uninstalling Pillow-9.4.0:
      Successfully uninstalled Pillow-9.4.0

requirements: AutoUpdate success ✅ 14.8s, installed 3 packages: ['gitpython>=3.1.30', 'pillow>=10.3.0', 'requests>=2.32.0']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect



YOLOv5 🚀 2024-7-13 Python-3.10.12 torch-2.3.0+cu121 CPU

100%|██████████| 166M/166M [00:01<00:00, 161MB/s]

Fusing layers... 
YOLOv5x summary: 444 layers, 86705005 parameters, 0 gradients, 205.5 GFLOPs
Adding AutoShape... 


AttributeError: 'NoneType' object has no attribute 'shape'